In [4]:
!pip install pyspark scikit-learn fastapi uvicorn nest-asyncio pyngrok joblib

# =========================
# IMPORTS
# =========================
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, avg, pandas_udf
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType

import pandas as pd
import numpy as np
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# =========================
# START SPARK
# =========================
spark = SparkSession.builder.appName("AirCargo").getOrCreate()

# =========================
# STEP 1: LOAD DATA
# =========================
df = spark.read.option("delimiter", "|") \
               .option("header", True) \
               .csv("/content/.config/Air_Cargo/Air_Cargo_Data")


df = df.withColumn("weight", col("weight").cast("int"))

print("Data Loaded")
df.show()

# =========================
# STEP 2: ADD SYNTHETIC FEATURES (PANDAS PART)
# =========================
pdf = df.toPandas()

# Add artificial fields for ML richness
np.random.seed(42)
pdf["retry_count"] = np.random.randint(0, 5, size=len(pdf))
pdf["processing_time"] = np.random.uniform(1, 10, size=len(pdf))
pdf["payload_size"] = pdf["weight"]

# YOUR LOGIC (status creation)
pdf["status"] = (
    (pdf["retry_count"] > 2) |
    (pdf["processing_time"] > 5)
).astype(int)

# Save dataset
import os
os.makedirs("data", exist_ok=True)
pdf.to_csv("data/cargo.csv", index=False)

print("Pandas dataset created")

# =========================
# STEP 3: BACK TO SPARK
# =========================
df = spark.createDataFrame(pdf)

# Feature 1: High retry
df = df.withColumn(
    "high_retry",
    when(col("retry_count") > 2, 1).otherwise(0)
)

# Feature 2: Rolling latency
windowSpec = Window.partitionBy("export_origin").rowsBetween(-5, 0)

df = df.withColumn(
    "rolling_latency",
    avg("processing_time").over(windowSpec)
)
# Feature 3
df = df.withColumn(
    "high_retry",
    when(col("retry_count") > 2, 1).otherwise(0)
)

# Feature 4
windowSpec = Window.partitionBy("export_origin").rowsBetween(-5, 0)

df = df.withColumn(
    "rolling_latency",
    avg("processing_time").over(windowSpec)
)
from pyspark.sql.functions import count, sum

windowSpec2 = Window.partitionBy("export_origin")

df = df.withColumn(
    "airport_congestion",
    count("*").over(windowSpec2)
)

df = df.withColumn(
    "total_cargo_load",
    sum("weight").over(windowSpec2)
)

# Save feature store
df.write.mode("overwrite").parquet("data/features")


print("Features created & stored")

# =========================
# STEP 4: MODEL TRAINING
# =========================
pdf = df.toPandas()

features = [
    "payload_size",
    "retry_count",
    "processing_time",
    "rolling_latency"
]

X = pdf[features]
y = pdf["status"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = RandomForestClassifier(n_estimators=200, max_depth=10)
model.fit(X_train, y_train)

os.makedirs("models", exist_ok=True)
joblib.dump(model, "models/model.pkl")

print("Model trained & saved")

# =========================
# STEP 5: SPARK BATCH INFERENCE (UDF)
# =========================

from pyspark.sql.functions import count
from pyspark.sql.window import Window
from pyspark.sql.functions import sum


model = joblib.load("models/model.pkl")

df = spark.read.parquet("data/features")

@pandas_udf(DoubleType())
def predict_udf(payload, retry, proc, latency):

    data = pd.DataFrame({
        "payload_size": payload,
        "retry_count": retry,
        "processing_time": proc,
        "rolling_latency": latency
    })

    preds = model.predict_proba(data)[:,1]
    return pd.Series(preds)

df = df.withColumn(
    "failure_probability",
    predict_udf(
        col("payload_size"),
        col("retry_count"),
        col("processing_time"),
        col("rolling_latency")
    )
)

print("Batch prediction done")
df.show()

# =========================
# STEP 6: FASTAPI (REAL-TIME API)
# =========================
from fastapi import FastAPI
from pyngrok import ngrok
import nest_asyncio
import uvicorn
import threading

app = FastAPI()

model = joblib.load("models/model.pkl")

@app.get("/")
def home():
    return {"msg": "AirCargo AI Running"}

@app.post("/predict")
def predict(payload_size: float, retry_count: int, processing_time: float, rolling_latency: float):

    data = np.array([
        [payload_size, retry_count, processing_time, rolling_latency]
    ])

    prob = model.predict_proba(data)[0][1]

    return {"failure_probability": float(prob)}

# Run API
nest_asyncio.apply()

def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run).start()




Data Loaded
+----------+-----------+-------------+-----------+---------------+----------------+------+
|awb_number|shipment_id|export_origin|destination|approval_status|      cargo_name|weight|
+----------+-----------+-------------+-----------+---------------+----------------+------+
|125-000001|     SHP001|          LHR|        CDG|       APPROVED|          Pharma|   250|
|125-000002|     SHP002|          DEL|        DXB|        PENDING|     Electronics|   180|
|125-000003|     SHP003|          JFK|        LHR|       REJECTED|     Perishables|   320|
|125-000004|     SHP004|          SIN|        DEL|       APPROVED|       Documents|   140|
|125-000005|     SHP005|          DXB|        JFK|        PENDING|Automobile Parts|   410|
|125-000006|     SHP006|          LHR|        FRA|       APPROVED|          Pharma|   290|
|125-000007|     SHP007|          DEL|        BOM|       REJECTED|     Electronics|   360|
|125-000008|     SHP008|          JFK|        CDG|        PENDING|     Perisha

INFO:     Started server process [10983]


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

NameError: name 'df' is not defined